# 01 Collecte des Données Biomédicales

## Objectif
Construire la base de connaissances du système RAG à partir d'abstracts PubMed sur la santé mentale des jeunes diplômés en recherche d'emploi.

## Population cible
**Jeunes diplômés universitaires** en phase de transition post-diplôme 
(recherche d'emploi, chômage initial) excluant explicitement les étudiants 
encore en cursus.

## Thématiques de collecte
| Thématique | Justification |
|------------|---------------|
| Santé mentale jeunes diplômés au chômage | Population cible exacte |
| Transition post-diplôme et santé mentale | Phase directement étudiée |
| Chômage et dépression/anxiété jeunes adultes | Outcomes principaux |
| Précarité emploi et santé mentale | Contexte post-diplôme |
| Revues systématiques sur le sujet | Synthèses existantes |

## Ce qu'on va produire
1. 500-1000 abstracts PubMed nettoyés et structurés
2. Dataset PubMedQA pour l'évaluation
3. Dataset MedQuAD filtré sur notre domaine

## Plan
1. Collecte via API PubMed (Entrez/Biopython)
2. Nettoyage et structuration
3. Téléchargement PubMedQA depuis Kaggle

In [2]:
# ============================================================
# 01 COLLECTE DES DONNÉES BIOMÉDICALES
# ============================================================

import pandas as pd
import numpy as np
import requests
import json
import time
import os
from pathlib import Path
from Bio import Entrez
import warnings
warnings.filterwarnings('ignore')

# Chargement des variables d'environnement
env_path = Path('..') / '.env'
with open(env_path, 'r', encoding='ascii') as f:
    for line in f:
        line = line.strip()
        if '=' in line and not line.startswith('#'):
            key, value = line.split('=', 1)
            os.environ[key.strip()] = value.strip()

# Configuration PubMed
Entrez.email = os.environ.get("PUBMED_EMAIL")
Entrez.tool  = "biomedical_rag_project"

print(f"Email Entrez : {Entrez.email}")
print(f"Clé Groq chargée : {'✅' if os.environ.get('GROQ_API_KEY') else '❌'}")

Email Entrez : mouwahiddorego@gmail.com
Clé Groq chargée : ✅


## Collecte des abstracts PubMed

### Stratégie de recherche
On définit 5 requêtes ciblées sur notre population (jeunes diplômés 
en transition post-diplôme) en excluant explicitement les études 
portant uniquement sur les étudiants en cursus.

### Période de collecte
2000-2025 pour capturer l'évolution récente du sujet.

### Volume cible
500-1000 abstracts.

In [3]:
requetes = {
    "graduates_unemployment_mental_health": (
        '(graduates[tiab] OR "university graduates"[tiab] OR '
        '"college graduates"[tiab]) AND '
        '(unemployment[tiab] OR "job seeking"[tiab] OR '
        '"job search"[tiab]) AND '
        '(mental health[tiab] OR depression[tiab] OR '
        'anxiety[tiab] OR wellbeing[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    ),
    "post_graduation_transition": (
        '("post-graduation"[tiab] OR "after graduation"[tiab] OR '
        '"school-to-work transition"[tiab] OR '
        '"labour market entry"[tiab]) AND '
        '(mental health[tiab] OR psychological[tiab] OR '
        'depression[tiab] OR anxiety[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    ),
    "youth_unemployment_depression": (
        '("young adults"[tiab] OR youth[tiab]) AND '
        '(unemployment[tiab] OR "not in employment"[tiab]) AND '
        '(depression[tiab] OR anxiety[tiab] OR '
        '"mental health"[tiab] OR distress[tiab]) AND '
        '(graduate[tiab] OR degree[tiab] OR educated[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    ),
    "precarious_employment_graduates": (
        '("precarious employment"[tiab] OR "job insecurity"[tiab] OR '
        '"underemployment"[tiab]) AND '
        '(graduates[tiab] OR "young workers"[tiab]) AND '
        '(mental health[tiab] OR depression[tiab] OR '
        'anxiety[tiab] OR stress[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    ),
    "systematic_review_unemployment_mental_health": (
        '(systematic review[pt] OR meta-analysis[pt]) AND '
        '(unemployment[tiab] OR "job loss"[tiab]) AND '
        '("mental health"[tiab] OR depression[tiab] OR anxiety[tiab]) AND '
        '("young adults"[tiab] OR graduates[tiab] OR youth[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    )
}

for nom, requete in requetes.items():
    print(f"\n📌 {nom}")
    print(f"   {requete[:100]}...")


📌 graduates_unemployment_mental_health
   (graduates[tiab] OR "university graduates"[tiab] OR "college graduates"[tiab]) AND (unemployment[tia...

📌 post_graduation_transition
   ("post-graduation"[tiab] OR "after graduation"[tiab] OR "school-to-work transition"[tiab] OR "labour...

📌 youth_unemployment_depression
   ("young adults"[tiab] OR youth[tiab]) AND (unemployment[tiab] OR "not in employment"[tiab]) AND (dep...

📌 precarious_employment_graduates
   ("precarious employment"[tiab] OR "job insecurity"[tiab] OR "underemployment"[tiab]) AND (graduates[...

📌 systematic_review_unemployment_mental_health
   (systematic review[pt] OR meta-analysis[pt]) AND (unemployment[tiab] OR "job loss"[tiab]) AND ("ment...


## Comptage des articles disponibles

Avant de télécharger, on vérifie combien d'articles correspondent 
à chaque requête. Cela permet de :
- Ajuster les requêtes si trop peu ou trop de résultats
- Estimer le temps de téléchargement
- Vérifier la pertinence de nos mots-clés

In [5]:
# ============================================================
# Comptage des articles disponibles par requête
# ============================================================

def compter_articles(requete, nom):
    """Compte le nombre d'articles sans les télécharger."""
    try:
        handle = Entrez.esearch(
            db="pubmed",
            term=requete,
            retmax=0  # On veut juste le compte
        )
        record = Entrez.read(handle)
        handle.close()
        count = int(record["Count"])
        return count
    except Exception as e:
        print(f"❌ Erreur pour {nom} : {e}")
        return 0

print("=== COMPTAGE DES ARTICLES DISPONIBLES ===\n")
total = 0
counts = {}

for nom, requete in requetes.items():
    count = compter_articles(requete, nom)
    counts[nom] = count
    total += count
    print(f"📌 {nom}")
    print(f"   → {count:,} articles trouvés\n")
    time.sleep(0.5)  
    
print(f"{'='*50}")
print(f"Total brut (avec doublons potentiels) : {total:,} articles")

=== COMPTAGE DES ARTICLES DISPONIBLES ===

📌 graduates_unemployment_mental_health
   → 31 articles trouvés

📌 post_graduation_transition
   → 193 articles trouvés

📌 youth_unemployment_depression
   → 17 articles trouvés

📌 precarious_employment_graduates
   → 7 articles trouvés

📌 systematic_review_unemployment_mental_health
   → 10 articles trouvés

Total brut (avec doublons potentiels) : 258 articles


258 articles au total, c'est un peu moins que ce qu'on espérait (500-1000). On va donc élargi un peu notre lexique de recherche.

In [6]:
# ============================================================
# Requêtes élargies avec synonymes supplémentaires
# ============================================================

requetes_v2 = {
    "graduates_unemployment_mental_health": (
        '(graduates[tiab] OR "university graduates"[tiab] OR '
        '"college graduates"[tiab] OR "recent graduates"[tiab] OR '
        '"newly graduated"[tiab] OR "first job"[tiab]) AND '
        '(unemployment[tiab] OR "job seeking"[tiab] OR '
        '"job search"[tiab] OR "job hunting"[tiab] OR '
        '"looking for work"[tiab]) AND '
        '(mental health[tiab] OR depression[tiab] OR '
        'anxiety[tiab] OR wellbeing[tiab] OR '
        '"psychological distress"[tiab] OR burnout[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    ),
    "post_graduation_transition": (
        '("post-graduation"[tiab] OR "after graduation"[tiab] OR '
        '"school-to-work transition"[tiab] OR '
        '"labour market entry"[tiab] OR "labor market entry"[tiab] OR '
        '"school to work"[tiab] OR "university to work"[tiab] OR '
        '"first employment"[tiab]) AND '
        '(mental health[tiab] OR psychological[tiab] OR '
        'depression[tiab] OR anxiety[tiab] OR '
        '"psychological distress"[tiab] OR wellbeing[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    ),
    "youth_unemployment_depression": (
        '("young adults"[tiab] OR youth[tiab] OR '
        '"young people"[tiab] OR "young workers"[tiab]) AND '
        '(unemployment[tiab] OR "not in employment"[tiab] OR '
        'NEET[tiab] OR "jobless"[tiab] OR "out of work"[tiab]) AND '
        '(depression[tiab] OR anxiety[tiab] OR '
        '"mental health"[tiab] OR distress[tiab] OR '
        '"mental disorder"[tiab] OR "psychological distress"[tiab]) AND '
        '(graduate[tiab] OR degree[tiab] OR educated[tiab] OR '
        'university[tiab] OR college[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    ),
    "precarious_employment_graduates": (
        '("precarious employment"[tiab] OR "job insecurity"[tiab] OR '
        '"underemployment"[tiab] OR "temporary employment"[tiab] OR '
        '"unstable employment"[tiab] OR "overeducation"[tiab]) AND '
        '(graduates[tiab] OR "young workers"[tiab] OR '
        '"educated workers"[tiab] OR "higher education"[tiab]) AND '
        '(mental health[tiab] OR depression[tiab] OR '
        'anxiety[tiab] OR stress[tiab] OR wellbeing[tiab] OR '
        '"psychological distress"[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    ),
    "systematic_review_unemployment_mental_health": (
        '(systematic review[pt] OR meta-analysis[pt]) AND '
        '(unemployment[tiab] OR "job loss"[tiab] OR '
        '"job insecurity"[tiab] OR NEET[tiab]) AND '
        '("mental health"[tiab] OR depression[tiab] OR '
        'anxiety[tiab] OR "psychological distress"[tiab] OR '
        'wellbeing[tiab]) AND '
        '("young adults"[tiab] OR graduates[tiab] OR '
        'youth[tiab] OR "young people"[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    ),
    "employment_status_mental_health_graduates": (
        '("employment status"[tiab] OR "labour market"[tiab] OR '
        '"labor market"[tiab] OR "job market"[tiab]) AND '
        '("mental health"[tiab] OR depression[tiab] OR '
        'anxiety[tiab] OR "psychological distress"[tiab] OR '
        'wellbeing[tiab] OR burnout[tiab]) AND '
        '(graduates[tiab] OR "young adults"[tiab] OR '
        'youth[tiab] OR "higher education"[tiab]) '
        'AND ("2000"[PDat]:"2025"[PDat])'
    )
}

print("=== COMPTAGE V2 REQUÊTES ÉLARGIES ===\n")
total_v2 = 0
counts_v2 = {}

for nom, requete in requetes_v2.items():
    count = compter_articles(requete, nom)
    counts_v2[nom] = count
    total_v2 += count
    print(f"📌 {nom}")
    print(f"   → {count:,} articles trouvés\n")
    time.sleep(0.5)

print(f"{'='*50}")
print(f"Total V1 : {total:,} articles")
print(f"Total V2 : {total_v2:,} articles")
print(f"Gain     : +{total_v2 - total:,} articles")

=== COMPTAGE V2 REQUÊTES ÉLARGIES ===

📌 graduates_unemployment_mental_health
   → 37 articles trouvés

📌 post_graduation_transition
   → 278 articles trouvés

📌 youth_unemployment_depression
   → 37 articles trouvés

📌 precarious_employment_graduates
   → 28 articles trouvés

📌 systematic_review_unemployment_mental_health
   → 16 articles trouvés

📌 employment_status_mental_health_graduates
   → 291 articles trouvés

Total V1 : 258 articles
Total V2 : 687 articles
Gain     : +429 articles


## Téléchargement des abstracts

On va téléchargé les abstracts de toutes les requêtes V2.
Paramètres :
- **retmax = 300** par requête maximum
- **Pause de 0.34s** entre chaque requête (limite API PubMed : 3 req/sec)
- **Format XML** → parsing via Biopython

In [7]:
# ============================================================
# Téléchargement des abstracts PubMed
# ============================================================

def telecharger_abstracts(requete, nom, retmax=300):
    """
    Télécharge les abstracts PubMed pour une requête donnée.
    Retourne une liste de dictionnaires avec les métadonnées.
    """
    articles = []
    
    try:
        # Étape 1 : récupérer les IDs des articles
        handle = Entrez.esearch(
            db="pubmed",
            term=requete,
            retmax=retmax
        )
        record  = Entrez.read(handle)
        handle.close()
        ids     = record["IdList"]
        
        if not ids:
            print(f"  ⚠️ Aucun article pour {nom}")
            return []
        
        print(f"  → {len(ids)} IDs récupérés")
        
        # Étape 2 : télécharger les détails par batch de 50
        batch_size = 50
        for i in range(0, len(ids), batch_size):
            batch_ids = ids[i:i+batch_size]
            
            handle = Entrez.efetch(
                db="pubmed",
                id=batch_ids,
                rettype="xml",
                retmode="xml"
            )
            records = Entrez.read(handle)
            handle.close()
            
            # Extraction des informations
            for article in records["PubmedArticle"]:
                try:
                    # Titre
                    titre = str(article["MedlineCitation"]["Article"]["ArticleTitle"])
                    
                    # Abstract
                    abstract_data = article["MedlineCitation"]["Article"].get("Abstract", {})
                    abstract_texts = abstract_data.get("AbstractText", [])
                    
                    if isinstance(abstract_texts, list):
                        abstract = " ".join([str(t) for t in abstract_texts])
                    else:
                        abstract = str(abstract_texts)
                    
                    # PMID
                    pmid = str(article["MedlineCitation"]["PMID"])
                    
                    # Année de publication
                    pub_date = article["MedlineCitation"]["Article"].get(
                        "Journal", {}
                    ).get("JournalIssue", {}).get("PubDate", {})
                    annee = str(pub_date.get("Year", "Unknown"))
                    
                    # Journal
                    journal = str(
                        article["MedlineCitation"]["Article"]
                        .get("Journal", {})
                        .get("Title", "Unknown")
                    )
                    
                    # On garde uniquement les articles avec un abstract
                    if abstract and len(abstract) > 50:
                        articles.append({
                            "pmid"    : pmid,
                            "titre"   : titre,
                            "abstract": abstract,
                            "annee"   : annee,
                            "journal" : journal,
                            "requete" : nom
                        })
                
                except Exception as e:
                    continue
            
            time.sleep(0.34)  # Respect limite API PubMed
        
        return articles
    
    except Exception as e:
        print(f"  ❌ Erreur : {e}")
        return []

# ============================================================
# Lancement de la collecte
# ============================================================

tous_articles = []

for nom, requete in requetes_v2.items():
    print(f" {nom}")
    articles = telecharger_abstracts(requete, nom, retmax=300)
    tous_articles.extend(articles)
    print(f" {len(articles)} articles collectés\n")
    time.sleep(1)

print(f"{'='*50}")
print(f"Total collecté : {len(tous_articles):,} articles")

 graduates_unemployment_mental_health
  → 37 IDs récupérés
 37 articles collectés

 post_graduation_transition
  → 278 IDs récupérés
 278 articles collectés

 youth_unemployment_depression
  → 37 IDs récupérés
 37 articles collectés

 precarious_employment_graduates
  → 28 IDs récupérés
 28 articles collectés

 systematic_review_unemployment_mental_health
  → 16 IDs récupérés
 16 articles collectés

 employment_status_mental_health_graduates
  → 291 IDs récupérés
 288 articles collectés

Total collecté : 684 articles


## Nettoyage et dédoublonnage

684 articles bruts collectés avec potentiels des doublons (un article peut apparaître dans plusieurs requêtes).

On dédoublonne sur le PMID qui est l'identifiant unique de PubMed. On va nettoyé également les abstracts (caractères spéciaux, espaces).

In [8]:
# ============================================================
# Nettoyage et dédoublonnage
# ============================================================

import re

# Création du dataframe
df_raw = pd.DataFrame(tous_articles)

print(f"=== AVANT NETTOYAGE ===")
print(f"Total articles     : {len(df_raw):,}")
print(f"Doublons (PMID)    : {df_raw.duplicated('pmid').sum():,}")
print(f"Sans abstract      : {(df_raw['abstract'] == '').sum():,}")
print(f"\nDistribution par requête :")
print(df_raw['requete'].value_counts())

# Dédoublonnage sur PMID
df_clean = df_raw.drop_duplicates(subset='pmid', keep='first')

# Nettoyage des textes
def nettoyer_texte(texte):
    """Nettoie un texte : supprime caractères spéciaux, espaces multiples."""
    if not isinstance(texte, str):
        return ""
    # Suppression des balises HTML/XML résiduelles
    texte = re.sub(r'<[^>]+>', ' ', texte)
    # Suppression des caractères spéciaux sauf ponctuation utile
    texte = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)]', ' ', texte)
    # Suppression des espaces multiples
    texte = re.sub(r'\s+', ' ', texte).strip()
    return texte

df_clean['titre']    = df_clean['titre'].apply(nettoyer_texte)
df_clean['abstract'] = df_clean['abstract'].apply(nettoyer_texte)

# Suppression des abstracts trop courts (< 100 caractères)
df_clean = df_clean[df_clean['abstract'].str.len() >= 100]

# Réinitialisation de l'index
df_clean = df_clean.reset_index(drop=True)

print(f"\n=== APRÈS NETTOYAGE ===")
print(f"Articles uniques   : {len(df_clean):,}")
print(f"Doublons supprimés : {len(df_raw) - len(df_clean):,}")
print(f"\nDistribution par année :")
print(df_clean['annee'].value_counts().sort_index().tail(10))
print(f"\nAperçu :")
print(df_clean[['pmid', 'titre', 'annee', 'journal']].head(5))

=== AVANT NETTOYAGE ===
Total articles     : 684
Doublons (PMID)    : 31
Sans abstract      : 0

Distribution par requête :
requete
employment_status_mental_health_graduates       288
post_graduation_transition                      278
graduates_unemployment_mental_health             37
youth_unemployment_depression                    37
precarious_employment_graduates                  28
systematic_review_unemployment_mental_health     16
Name: count, dtype: int64

=== APRÈS NETTOYAGE ===
Articles uniques   : 652
Doublons supprimés : 32

Distribution par année :
annee
2018       28
2019       33
2020       46
2021       50
2022       59
2023       73
2024       73
2025       99
2026        8
Unknown     2
Name: count, dtype: int64

Aperçu :
       pmid                                              titre annee  \
0  40160383  Exploring prevalence and factors associated wi...  2025   
1  40077974  Employment Trajectories of Recently Certified ...  2025   
2  39972449  Exploring the unemp

On observe que la distribution temporelle est très intéressante.

Une Forte concentration sur 2022-2025 (304 articles sur 652), le sujet est très actuel. 99 articles en 2025, les données sont très récentes. 8 articles en 2026, ce qui veut dire probablement que PubMed indexe déjà des articles à parution anticipée.

## Statistiques finales et sauvegarde

On a 652 articles uniques et propres qui sont prêts pour l'indexation RAG.
On analyse aussi la longueur des abstracts pour anticiper la stratégie de chunking.

In [10]:
# ============================================================
# Statistiques descriptives + sauvegarde
# ============================================================

import matplotlib.pyplot as plt

# Longueur des abstracts
df_clean['abstract_length'] = df_clean['abstract'].str.len()
df_clean['abstract_words']  = df_clean['abstract'].str.split().str.len()

print("=== STATISTIQUES DES ABSTRACTS ===")
print(f"Longueur moyenne   : {df_clean['abstract_length'].mean():.0f} caractères")
print(f"Longueur médiane   : {df_clean['abstract_length'].median():.0f} caractères")
print(f"Longueur min       : {df_clean['abstract_length'].min():,} caractères")
print(f"Longueur max       : {df_clean['abstract_length'].max():,} caractères")
print(f"\nMots moyens        : {df_clean['abstract_words'].mean():.0f} mots")
print(f"Mots médians       : {df_clean['abstract_words'].median():.0f} mots")

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution longueur abstracts
axes[0].hist(df_clean['abstract_length'], bins=30, 
             color='steelblue', alpha=0.8)
axes[0].set_title('Distribution longueur abstracts\n(caractères)')
axes[0].set_xlabel('Caractères')
axes[0].set_ylabel('Nombre d\'articles')

# Distribution par année
annees = df_clean[df_clean['annee'] != 'Unknown']['annee'].value_counts().sort_index()
axes[1].bar(annees.index, annees.values, color='darkorange', alpha=0.8)
axes[1].set_title('Distribution par année')
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Nombre d\'articles')
axes[1].tick_params(axis='x', rotation=45)

# Distribution par requête
requete_counts = df_clean['requete'].value_counts()
axes[2].barh(requete_counts.index, requete_counts.values, 
             color='green', alpha=0.8)
axes[2].set_title('Distribution par requête')
axes[2].set_xlabel('Nombre d\'articles')

plt.tight_layout()
plt.show()

# Sauvegarde
output_path = Path('../data/processed/pubmed_articles.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False, encoding='utf-8')

print(f"\n Dataset sauvegardé : data/processed/pubmed_articles.csv")
print(f"Shape final : {df_clean.shape}")
print(f"Colonnes    : {list(df_clean.columns)}")

ModuleNotFoundError: No module named 'matplotlib'